## Import libraries

In [1]:
!pip install --no-index /kaggle/input/notebooks/packagemanager/pm-111391774-at-03-06-2026-10-44-29/portalocker-3.2.0-py3-none-any.whl

!pip install --no-index /kaggle/input/notebooks/packagemanager/pm-111391774-at-03-06-2026-10-44-29/sacrebleu-2.6.0-py3-none-any.whl

Processing /kaggle/input/notebooks/packagemanager/pm-111391774-at-03-06-2026-10-44-29/portalocker-3.2.0-py3-none-any.whl
Processing /kaggle/input/notebooks/packagemanager/pm-111391774-at-03-06-2026-10-44-29/sacrebleu-2.6.0-py3-none-any.whl


In [2]:
import os
import re
import gc
import json
import math
import time
import glob
import logging
import difflib
from dataclasses import dataclass, asdict
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from tqdm.auto import tqdm
from datetime import datetime
from pathlib import Path

import sacrebleu


## Enviroment

In [3]:
# ---------------------------
# 0) Environment knobs
# ---------------------------
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "0")
os.environ.setdefault("TORCH_CUDNN_V8_API_ENABLED", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("byt-robust")

def print_environment_info():
    logger.info(f"torch={torch.__version__}")
    logger.info(f"cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"gpu={torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        logger.info(f"gpu_mem_total_gb={props.total_memory/1024**3:.2f}")
    # BetterTransformer availability (optional)
    try:
        from optimum.bettertransformer import BetterTransformer  # noqa: F401
        logger.info("optimum.bettertransformer is available (will try to use if enabled).")
    except Exception:
        logger.info("optimum.bettertransformer not available (will skip).")

print_environment_info()

2026-03-22 13:15:47,056 | INFO | torch=2.8.0+cu126
2026-03-22 13:15:47,301 | INFO | cuda_available=True
2026-03-22 13:15:47,340 | INFO | gpu=Tesla T4
2026-03-22 13:15:47,341 | INFO | gpu_mem_total_gb=14.56
2026-03-22 13:15:47,342 | INFO | optimum.bettertransformer not available (will skip).


## Configs

In [4]:
@dataclass
class CFG:
    # =========================
    # PATHS
    # =========================
    test_path: str = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    train_path: str = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"

    # Single-model (fallback)
    model_path: str = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"

    # Legacy two-model fields (kept for backward compatibility)
    model_a_path: Optional[str] = None
    model_b_path: Optional[str] = None
    model_c_path: Optional[str] = None

    output_dir: str = "/kaggle/working"
    temp_model_dir: str = "/kaggle/working/_models"

    # =========================
    # TOKENIZATION / LOADING
    # =========================
    max_length: int = 512
    batch_size: int = 8
    num_workers: int = 2
    pin_memory: bool = True

    # =========================
    # GENERATION
    # =========================
    num_beams: int = 8
    max_new_tokens: int = 512
    length_penalty: float = 1.3
    early_stopping: bool = True
    no_repeat_ngram_size: int = 0
    repetition_penalty: Optional[float] = None

    num_beam_cands: int = 4
    num_sample_cands: int = 6

    # sampling
    do_sample: bool = True
    top_p: float = 0.9
    temperature: float = 0.7

    # bucket sampler
    use_bucket_batching: bool = True
    num_buckets: int = 32

    # =========================
    # MBR
    # =========================
    mbr_pool_cap: int = 64

    # =========================
    # PERFORMANCE
    # =========================
    use_mixed_precision: bool = True
    use_better_transformer: bool = True
    use_bucket_batching: bool = True
    use_adaptive_beams: bool = True
    use_vectorized_postproc: bool = True

    # =========================
    # RUNTIME
    # =========================
    checkpoint_freq: int = 2000
    empty_cache_every: int = 10
    validate_samples: int = 6

    # =========================
    # POSTPROCESS
    # =========================
    postprocess_mode: str = "safe_then_conditional_aggressive"
    aggressive_trigger_badness: float = 1.5
    min_words_fallback: int = 3
    default_translation: str = "The tablet is too damaged to translate."

    # =========================
    # DEVICE
    # =========================
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    def __post_init__(self):
        if self.device == "cpu":
            self.use_mixed_precision = False
            self.use_better_transformer = False
            self.pin_memory = False


@dataclass
class ModelRunSpec:
    label: str
    model_path: str
    prompt_template: str = "{text}"
    preset: str = "baseline"
    submission_name: Optional[str] = None
    cfg: Optional[CFG] = None


## Preprocessor

Includes

- small gap, big gap convertor
- normalize spaces

In [5]:
V2_RE = re.compile(r"([aAeEiIuU])(?:2|\u2082)")
V3_RE = re.compile(r"([aAeEiIuU])(?:3|\u2083)")
ACUTE = str.maketrans({"a": "a", "e": "e", "i": "i", "u": "u", "A": "A", "E": "E", "I": "I", "U": "U"})
GRAVE = str.maketrans({"a": "a", "e": "e", "i": "i", "u": "u", "A": "A", "E": "E", "I": "I", "U": "U"})


def ascii_to_diacritics(text: str) -> str:
    s = str(text or "")
    s = s.replace("sz", "sh").replace("SZ", "Sh")
    s = s.replace("s,", "s").replace("S,", "S")
    s = s.replace("t,", "t").replace("T,", "T")
    s = V2_RE.sub(lambda m: m.group(1).translate(ACUTE), s)
    s = V3_RE.sub(lambda m: m.group(1).translate(GRAVE), s)
    return s


ALLOWED_FRACS = [
    (1 / 6, "0.16666"),
    (1 / 4, "0.25"),
    (1 / 3, "0.33333"),
    (1 / 2, "0.5"),
    (2 / 3, "0.66666"),
    (3 / 4, "0.75"),
    (5 / 6, "0.83333"),
]
FRAC_TOL = 2e-3
FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
WS_RE = re.compile(r"\s+")


def canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|\u2026+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I,
)

CHAR_TRANS = str.maketrans({
    "\u1e2b": "h",
    "\u1e2a": "H",
    "\u02be": "",
    "\u2080": "0",
    "\u2081": "1",
    "\u2082": "2",
    "\u2083": "3",
    "\u2084": "4",
    "\u2085": "5",
    "\u2086": "6",
    "\u2087": "7",
    "\u2088": "8",
    "\u2089": "9",
    "\u2014": "-",
    "\u2013": "-",
})
SUB_X = "\u2093"

DET_UPPER_RE = re.compile(r"\(([A-Z0-9]{1,6})\)")
DET_LOWER_RE = re.compile(r"\(([a-z]{1,4})\)")
PN_RE = re.compile(r"\bPN\b")
SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)(?:\.\s*(?:plur|plural|sing|singular))?\.?\s*[^)]*\)",
    re.I,
)
BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)
REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")
FORBIDDEN_TRANS = str.maketrans("", "", "()<>[]+;")
MULTI_GAP_RE = re.compile(r"(?:<gap>\s*){2,}")

class Preprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts, dtype="object").fillna("").astype(str)

        ser = ser.apply(ascii_to_diacritics)

        ser = ser.str.replace(DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(DET_LOWER_RE, r"{\1}", regex=True)

        # GAP handling (safe)
        ser = ser.str.replace(GAP_UNIFIED_RE, "<gap>", regex=True)
        ser = ser.str.replace(r"(?:<gap>\s*){2,}", "<big_gap> ", regex=True)

        ser = ser.str.translate(CHAR_TRANS)
        ser = ser.str.replace(SUB_X, "", regex=False)

        # ⚠ optional (disable if unsure)
        ser = ser.str.replace(FLOAT_RE, lambda m: canon_decimal(float(m.group(1))), regex=True)

        ser = ser.str.replace(WS_RE, " ", regex=True).str.strip()

        return ser.tolist()

## Postprocessor

### Safe Postprocessor

Only contain cleaning format, not editing contents (rewrite sentences)

In [6]:
class SafePostprocessor:
    """
    Conservative cleanup.
    - Preserve common punctuation and formatting.
    - Protect <gap>/<big_gap>.
    - Normalize a few known characters.
    """
    SUBSCRIPT_TO_NORMAL = str.maketrans({
        "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4","₅":"5","₆":"6","₇":"7","₈":"8","₉":"9"
    })

    def __init__(self, use_unicode_fractions: bool = False, strip_dash_old: bool = False):
        self.use_unicode_fractions = use_unicode_fractions
        self.strip_dash_old = strip_dash_old

        self.forbidden_chars = re.compile(r"[⌈⌋⌊⌉]")
        self.multi_space = re.compile(r"\s+")
        self.space_before_punct = re.compile(r"\s+([,.;:!?])")
        self.multi_punct = re.compile(r"([!?.,])\1{2,}")
        self.dashes = re.compile(r"[–—]")

        self.prot_gap = "\uFFF0"
        self.prot_big = "\uFFF1"

        self.bracket_x = re.compile(r"\[\s*x\s*\]|\(\s*x\s*\)", re.IGNORECASE)
        self.bare_x = re.compile(r"(?:(?<=\s)|^)x(?=(\s|$))", re.IGNORECASE)
        self.ellipsis = re.compile(r"(\.\.\.+|…+)")

        self.frac_map = {
            "0.5": "½", "0.25": "¼", "0.75": "¾",
            "1/2": "½", "1/4": "¼", "3/4": "¾",
        }
        self.frac_re = re.compile(r"\b(0\.5|0\.25|0\.75|1/2|1/4|3/4)\b")

    def postprocess_one(self, text: str) -> str:
        if text is None:
            text = ""
        t = str(text)

        t = t.replace("ḫ", "h").replace("Ḫ", "H")
        t = t.translate(self.SUBSCRIPT_TO_NORMAL)
        t = self.dashes.sub("-", t)

        t = self.bracket_x.sub("<gap>", t)
        t = self.bare_x.sub("<gap>", t)
        t = self.ellipsis.sub("<big_gap>", t)

        t = t.replace("<gap>", self.prot_gap).replace("<big_gap>", self.prot_big)
        t = self.forbidden_chars.sub("", t)
        t = t.replace(self.prot_gap, "<gap>").replace(self.prot_big, "<big_gap>")

        if self.strip_dash_old:
            t = t.strip(" -")

        if self.use_unicode_fractions:
            t = self.frac_re.sub(lambda m: self.frac_map.get(m.group(1), m.group(1)), t)

        t = self.space_before_punct.sub(r"\1", t)
        t = self.multi_punct.sub(r"\1", t)
        t = self.multi_space.sub(" ", t).strip()
        return t

    def postprocess_batch(self, texts: List[str]) -> List[str]:
        return [self.postprocess_one(x) for x in texts]


### Aggressive Postprocessor

Use for bad translation (badness >= threshold), include:

- Remove grammatical notes
- Remove repeated words
- Remove repeated phrases (n-grams)
- Consolidate gaps

In [7]:
import re
import pandas as pd
from typing import List, Union

class AggressivePostprocessor:
    def __init__(self, ngram_dedupe_max_n: int = 4):
        self.ngram_dedupe_max_n = ngram_dedupe_max_n

        # 1. Gap and Annotation Patterns
        self.pat_gap = re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I)
        self.pat_big_gap = re.compile(r"(\.{3,}|…|\[\.+\])")
        self.pat_gap_runs = re.compile(r"(?:<gap>\s*){2,}")
        self.pat_biggap_runs = re.compile(r"(?:<big_gap>\s*){2,}")
        self.pat_annotations = re.compile(
            r"\((fem|plur|pl|sing|singular|plural|masc|uncertain|\?|!|damaged|broken)\.?\s*\w*\)", 
            re.I
        )

        # 2. Deduplication and Punctuation Patterns
        self.pat_repeat_word = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)
        self.pat_punct_space = re.compile(r"\s+([.,:;!?])")
        self.pat_multi_punct = re.compile(r"([.,!?])\1+")
        self.pat_whitespace = re.compile(r"\s+")

        # 3. Translation Tables
        self.subscript_trans = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
        self.special_chars_trans = str.maketrans("ḫḪ", "hH")
        # Characters to remove (excluding what's inside <gap> tags)
        self.forbidden_chars = '!?()"—–<>⌈⌋⌊⌉[]+ʾ/;'
        self.forbidden_trans = str.maketrans("", "", self.forbidden_chars)

        # 4. Fraction Lookups (Regex, Replacement)
        self.fractions = [
            (re.compile(r"(\d+)\.5\b"), r"\1 ½"), (re.compile(r"\b0\.5\b"), "½"),
            (re.compile(r"(\d+)\.25\b"), r"\1 ¼"), (re.compile(r"\b0\.25\b"), "¼"),
            (re.compile(r"(\d+)\.75\b"), r"\1 ¾"), (re.compile(r"\b0\.75\b"), "¾"),
            (re.compile(r"(\d+)\.33+\d*\b"), r"\1 ⅓"), (re.compile(r"\b0\.33+\d*\b"), "⅓"),
            (re.compile(r"(\d+)\.66+\d*\b"), r"\1 ⅔"), (re.compile(r"\b0\.66+\d*\b"), "⅔"),
        ]

    def _dedupe_ngrams(self, text: str) -> str:
        """Removes recursive loops like 'king of king of' -> 'king of'."""
        tokens = text.split()
        if len(tokens) < 5:
            return text
        
        for n in range(self.ngram_dedupe_max_n, 1, -1):
            i = 0
            while i <= len(tokens) - 2 * n:
                if tokens[i : i + n] == tokens[i + n : i + 2 * n]:
                    del tokens[i + n : i + 2 * n]
                else:
                    i += 1
        return " ".join(tokens)

    def postprocess_one(self, text: str) -> str:
        """The core cleaning logic for a single string."""
        if not isinstance(text, str) or not text.strip():
            return ""

        # Phase 1: Normalization (Subscripts and Special Chars)
        t = text.translate(self.special_chars_trans)
        t = t.translate(self.subscript_trans)

        # Phase 2: Gap Standardization
        t = self.pat_gap.sub("<gap>", t)
        t = self.pat_big_gap.sub("<big_gap>", t)
        t = t.replace("<gap> <gap>", "<big_gap>")
        t = self.pat_gap_runs.sub("<big_gap> ", t)
        t = self.pat_biggap_runs.sub("<big_gap> ", t)

        # Phase 3: Domain Specifics (Fractions & Annotations)
        for pat, repl in self.fractions:
            t = pat.sub(repl, t)
        t = self.pat_annotations.sub("", t)

        # Phase 4: Forbidden Character Removal (Masking Tags)
        # We replace tags with placeholders so translate() doesn't kill the <> or [] in them
        t = t.replace("<gap>", "\x00GAP\x00").replace("<big_gap>", "\x00BIG\x00")
        t = t.translate(self.forbidden_trans)
        t = t.replace("\x00GAP\x00", " <gap> ").replace("\x00BIG\x00", " <big_gap> ")

        # Phase 5: Deduplication (Single words + N-Grams)
        t = self.pat_repeat_word.sub(r"\1", t)
        t = self._dedupe_ngrams(t)

        # Phase 6: Punctuation & Whitespace Cleanup
        t = self.pat_punct_space.sub(r"\1", t)
        t = self.pat_multi_punct.sub(r"\1", t)
        t = self.pat_whitespace.sub(" ", t)
        
        return t.strip().strip("-").strip()

    def postprocess_batch(self, translations: List[str]) -> List[str]:
        """
        Process a list of translations. 
        Uses a list comprehension for readability.
        """
        return [self.postprocess_one(t) for t in translations]


## Scoring translated

### Badness score (lower = better)

Include

- empty
- not enough/too many words
- many gaps
- repetitive words
- repetitive phrases

In [8]:
def badness_score(text: str) -> float:
    """Lower is better."""
    if text is None:
        text = ""
    t = str(text).strip()
    if not t:
        return 10.0

    words = t.split()
    n = len(words)

    score = 0.0
    if n < 5:
        score += 3.0
    if n < 3:
        score += 3.0
    if n > 500:
        score += 2.0
    if n > 650:
        score += 3.0

    gaps = t.count("<gap>") + t.count("<big_gap>")
    if gaps > 6:
        score += (gaps - 6) * 0.35

    rep = 0
    for i in range(2, n):
        if words[i].lower() == words[i-1].lower() == words[i-2].lower():
            rep += 1
    score += rep * 0.75

    if n >= 20:
        bigrams = list(zip(words, words[1:]))
        uniq = len(set(bigrams))
        if uniq > 0:
            repetitiveness = 1.0 - (uniq / max(1, len(bigrams)))
            if repetitiveness > 0.35:
                score += (repetitiveness - 0.35) * 6.0

    return score


### Quality score (higher = better)

Calculates how natrual of the output

Includes:

- Ideal length / Acceptable length
- Capitalized
- Well punctuation
- Include keywords
- Penalize gaps
- Apply badness


In [9]:
# These words are more common in Akkadian translit.
KEYWORDS = {
    "tablet", "king", "city", "god", "silver", "gold", "temple", "house", "palace",
    "year", "son", "daughter", "brother", "mother", "father", "gave", "took", "sent",
    "received", "grain", "sheep", "oil", "wine"
}

In [10]:
def quality_score(text: str) -> float:
    if text is None:
        text = ""
    t = str(text).strip()
    
    if not t:
        return 0.0 # so bad

    words = t.split()
    n = len(words)
    score = 0.0
    if 80 <= n <= 120:
        score += 2.0 # +3 if 80 <= len <= 120, check below
    if 50 <= n <= 200:
        score += 1.0 # +1 if 50 <= len <= 200

    # formatting
    if t[0].isupper():
        score += 0.5 # Uppercase the first word check
    if t.endswith((".", "?", "!")):
        score += 0.5 # Sentence ending punctation

    # keyword hints
    lw = {w.strip(".,;:!?\"'()[]").lower() for w in words}
    hit = len(lw.intersection(KEYWORDS))
    score += min(2.0, 0.25 * hit) # +0.25 per keyword, 2 max

    # penalize too many gaps
    gaps = t.count("<gap>") + t.count("<big_gap>")
    score -= min(2.0, 0.15 * gaps) # -0.15 per gap, 2 max

    # penalize heavy repetition by badness
    score -= 0.5 * max(0.0, badness_score(t) - 1.0)
    return score    

### Fuse texts (Final decision)

In [11]:
class MBRSelector:
    def __init__(self, pool_cap: int = 32):
        self.pool_cap = pool_cap
        self._metric = None

        if sacrebleu is not None:
            try:
                self._metric = sacrebleu.metrics.CHRF(word_order=2)
            except TypeError:
                try:
                    self._metric = sacrebleu.metrics.CHRF()
                except Exception:
                    self._metric = None
            except Exception:
                self._metric = None

    @staticmethod
    def _normalize_candidate(x) -> str:
        if x is None:
            return ""
        if isinstance(x, float) and np.isnan(x):
            return ""
        return str(x).strip()

    @classmethod
    def _dedup(cls, xs):
        seen, out = set(), []
        for x in xs or []:
            x = cls._normalize_candidate(x)
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    @staticmethod
    def _token_f1(a: str, b: str) -> float:
        aw = a.lower().split()
        bw = b.lower().split()
        if not aw or not bw:
            return 0.0

        ac, bc = {}, {}
        for w in aw:
            ac[w] = ac.get(w, 0) + 1
        for w in bw:
            bc[w] = bc.get(w, 0) + 1

        overlap = sum(min(ac.get(w, 0), bc.get(w, 0)) for w in set(ac) | set(bc))
        if overlap <= 0:
            return 0.0

        precision = overlap / max(1, len(aw))
        recall = overlap / max(1, len(bw))
        if precision + recall == 0:
            return 0.0
        return 100.0 * (2 * precision * recall) / (precision + recall)

    @staticmethod
    def _char_ratio(a: str, b: str) -> float:
        return 100.0 * difflib.SequenceMatcher(None, a, b).ratio()

    def _pair_score(self, a: str, b: str) -> float:
        a, b = self._normalize_candidate(a), self._normalize_candidate(b)
        if not a or not b:
            return 0.0

        if self._metric is not None:
            try:
                return float(self._metric.sentence_score(a, [b]).score)
            except Exception:
                pass

        return 0.65 * self._token_f1(a, b) + 0.35 * self._char_ratio(a, b)

    @staticmethod
    def _fallback_pick(cands: List[str]) -> str:
        if not cands:
            return ""
        return max(
            cands,
            key=lambda t: (quality_score(t) - 0.6 * badness_score(t), -len(str(t)))
        )

    def pick(self, candidates):
        cands = self._dedup(candidates)
        if self.pool_cap:
            cands = cands[:self.pool_cap]

        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]

        best_i, best_s = 0, -1e9

        for i in range(n):
            try:
                mbr = sum(
                    self._pair_score(cands[i], cands[j]) for j in range(n) if j != i
                ) / max(1, n - 1)

                q = quality_score(cands[i])
                b = badness_score(cands[i])
                score = 1.0 * mbr + 0.8 * q - 0.6 * b
            except Exception as e:
                logger.warning(f"MBR scoring failed for candidate {i}: {e}")
                score = -1e9

            if score > best_s:
                best_s, best_i = score, i

        if best_s <= -1e8:
            return self._fallback_pick(cands)
        return cands[best_i]


## Dataset

loads dataset + preprocess

In [12]:
class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor: Preprocessor):
        self.ids = df["id"].astype("str").tolist()
        raw = df["transliteration"].astype("object").fillna("").tolist()
        self.raw_inputs = preprocessor.preprocess_batch(raw)
        self.inputs = [f"translate Akkadian to English: {x}" for x in self.raw_inputs]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.raw_inputs[i]


## Bucket Sampler

Purpose: Merge samples has similar length into 1 batch --> faster inference

In [13]:
class BucketSampler(Sampler[List[int]]):
    def __init__(self, texts: List[str], batch_size: int, num_buckets=32, shuffle=False):
        self.batch_size = batch_size
        self.shuffle = shuffle

        lengths = np.array([max(1, len(t.split())) for t in texts], dtype=np.int32) # Measure text len.
        order = np.argsort(lengths) # Sort by len.
        self.indices = order.tolist()
        
        # Create buckets
        self.num_buckets = max(1, num_buckets)
        self.buckets = np.array_split(self.indices, self.num_buckets)

        # Split into batches
        self.batches = []
        rng = np.random.default_rng(42)
        for b in self.buckets:
            b = list(b)
            if shuffle:
                rng.shuffle(b)
            for i in range(0, len(b), batch_size): # split to batches in every buckets
                chunk = b[i:i+batch_size]
                if len(chunk) > 0:
                    self.batches.append(chunk) # push into self.batches

            if shuffle:
                rng.shuffle(self.batches)

    def __iter__(self):
        # Iteration
        for batch in self.batches:
            yield batch
            
    def __len__(self):
        # return number of batches
        return len(self.batches)


## Inference Engine

Includes

- load translation model
- prepares inputs
- runs generation
- clean the output
- return the prediction

In [14]:
class ModelWrapper:
    def __init__(self, model_path: str, cfg: CFG, logger: logging.Logger, label: str, prompt_template: str = "{text}"):
        self.cfg = cfg
        self.logger = logger
        self.label = label
        self.prompt_template = prompt_template or "{text}"

        logger.info(f"[{label}] Loading from {model_path}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(cfg.device).eval()

        if cfg.device == "cuda":
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass

        n = sum(p.numel() for p in self.model.parameters())
        logger.info(f"[{label}] {n:,} parameters")

        if cfg.device == "cuda":
            used = torch.cuda.memory_allocated() / 1e9
            logger.info(f"[{label}] GPU mem used: {used:.2f} GB")

        if cfg.use_better_transformer and cfg.device == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info(f"[{label}] BetterTransformer applied")
            except Exception as e:
                logger.warning(f"[{label}] BetterTransformer skipped: {e}")

    def _format_prompt(self, text: str) -> str:
        raw = str(text or "").strip()
        template = self.prompt_template or "{text}"
        if "{text}" in template:
            return template.format(text=raw)
        return f"{template}{raw}" if template else raw

    def collate(self, batch_samples):
        ids = [s[0] for s in batch_samples]
        texts = [self._format_prompt(s[1]) for s in batch_samples]

        enc = self.tokenizer(
            texts,
            max_length=self.cfg.max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return ids, enc

    def generate_candidates(self, input_ids, attention_mask, beam_size: int):
        cfg = self.cfg
        B = input_ids.shape[0]

        if cfg.use_mixed_precision and cfg.device == "cuda":
            ctx = torch.cuda.amp.autocast
        else:
            class _Null:
                def __enter__(self): return None
                def __exit__(self, *args): return False
            ctx = lambda: _Null()

        with ctx():
            nb = max(beam_size, cfg.num_beam_cands)

            beam_out = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                do_sample=False,
                num_beams=nb,
                num_return_sequences=cfg.num_beam_cands,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                repetition_penalty=cfg.repetition_penalty,
                no_repeat_ngram_size=cfg.no_repeat_ngram_size,
                use_cache=True,
            )

            beam_texts = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)

            samp_texts = []
            if cfg.num_sample_cands > 0:
                samp_out = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=True,
                    num_beams=1,
                    top_p=cfg.top_p,
                    temperature=cfg.temperature,
                    num_return_sequences=cfg.num_sample_cands,
                    max_new_tokens=cfg.max_new_tokens,
                    repetition_penalty=cfg.repetition_penalty,
                    use_cache=True,
                )

                samp_texts = self.tokenizer.batch_decode(samp_out, skip_special_tokens=True)

        Rb, Rs = cfg.num_beam_cands, cfg.num_sample_cands
        pools = []

        for i in range(B):
            p = list(beam_texts[i * Rb:(i + 1) * Rb])
            if Rs > 0:
                p += list(samp_texts[i * Rs:(i + 1) * Rs])
            pools.append(p)

        return pools

    def unload(self):
        label = self.label

        try:
            from optimum.bettertransformer import BetterTransformer
            self.model = BetterTransformer.reverse(self.model)
        except Exception:
            pass

        del self.model
        del self.tokenizer
        self.model = None
        self.tokenizer = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
            self.logger.info(f"[{label}] Unloaded. GPU free: {free:.2f} GB")


In [15]:
class EnsembleMBREngine:
    def __init__(self, cfg: CFG, logger: logging.Logger):
        self.cfg = cfg
        self.logger = logger
        self.preprocessor = Preprocessor()

        self.safe_pp = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=False)
        self.safe_pp_stripdash = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=True)
        self.agg_pp = AggressivePostprocessor()

        self.mbr = MBRSelector(pool_cap=cfg.mbr_pool_cap)

    def _best_effort_pick(self, candidates) -> str:
        cands = self.mbr._dedup(candidates)
        if not cands:
            return ""
        ranked = sorted(
            cands,
            key=lambda t: (quality_score(t) - 0.6 * badness_score(t), -len(str(t))),
            reverse=True,
        )
        return ranked[0]

    def _adaptive_beams(self, attn: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            return self.cfg.num_beams

        med = float(attn.sum(dim=1).float().median().item())
        short = max(self.cfg.num_beam_cands, self.cfg.num_beams // 2)
        return short if med < 100 else self.cfg.num_beams

    def _build_dataloader(self, dataset: AkkadianDataset, wrapper: ModelWrapper) -> DataLoader:
        if self.cfg.use_bucket_batching:
            sampler = BucketSampler(
                dataset.raw_inputs,
                self.cfg.batch_size,
                self.cfg.num_buckets
            )
            return DataLoader(
                dataset,
                batch_sampler=sampler,
                num_workers=self.cfg.num_workers,
                collate_fn=wrapper.collate,
                pin_memory=(self.cfg.device == "cuda"),
            )

        return DataLoader(
            dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            collate_fn=wrapper.collate,
            pin_memory=(self.cfg.device == "cuda"),
        )

    def _run_one_model(self, wrapper: ModelWrapper, dataset: AkkadianDataset) -> dict:
        dl = self._build_dataloader(dataset, wrapper)
        pools_by_id = {}

        with torch.inference_mode():
            for batch_ids, enc in tqdm(dl, desc=f"[{wrapper.label}]"):
                input_ids = enc.input_ids.to(self.cfg.device, non_blocking=True)
                attn = enc.attention_mask.to(self.cfg.device, non_blocking=True)
                beam_size = self._adaptive_beams(attn)

                try:
                    batch_pools = wrapper.generate_candidates(input_ids, attn, beam_size)
                    for sid, pool in zip(batch_ids, batch_pools):
                        pools_by_id[str(sid)] = pool

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        self.logger.error(f"OOM in [{wrapper.label}] — skipping batch")
                        torch.cuda.empty_cache()
                        for sid in batch_ids:
                            pools_by_id.setdefault(str(sid), [])
                    else:
                        raise

                except Exception as e:
                    self.logger.error(f"[{wrapper.label}] batch error: {e}")
                    for sid in batch_ids:
                        pools_by_id.setdefault(str(sid), [])

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        return pools_by_id

    def _postprocess(self, texts):
        if not texts:
            return []

        mode = self.cfg.postprocess_mode

        if mode == "safe_only":
            return self.safe_pp.postprocess_batch(texts)

        if mode == "aggressive_only":
            out = self.safe_pp_stripdash.postprocess_batch(texts)
            return self.agg_pp.postprocess_batch(out)

        safe = self.safe_pp.postprocess_batch(texts)

        return [
            self.agg_pp.postprocess_one(t)
            if badness_score(t) >= self.cfg.aggressive_trigger_badness
            else t
            for t in safe
        ]

    def _prepare_candidates(self, texts: List[str]) -> List[str]:
        texts = [str(x).strip() for x in texts if x and str(x).strip()]
        texts = list(dict.fromkeys(texts))
        texts = [x for x in texts if badness_score(x) < 8.0]
        pp = self._postprocess(texts)
        return self.mbr._dedup(pp or texts)

    def _pick_for_submission(self, texts: List[str]) -> str:
        prepared = self._prepare_candidates(texts)
        if not prepared:
            return self.cfg.default_translation
        try:
            chosen = self.mbr.pick(prepared)
        except Exception as e:
            self.logger.warning(f"Single-model MBR failed, using heuristic fallback: {e}")
            chosen = self._best_effort_pick(prepared)
        return chosen or self._best_effort_pick(prepared) or self.cfg.default_translation

    def _hybrid_pick(self, combined_candidates: List[str], model_best_texts: List[str]) -> str:
        prepared = self._prepare_candidates(combined_candidates)
        per_model = self._prepare_candidates(model_best_texts)

        mbr_choice = ""
        if prepared:
            try:
                mbr_choice = self.mbr.pick(prepared)
            except Exception as e:
                self.logger.warning(f"Global MBR failed; falling back to heuristic shortlist: {e}")

        fused_choice = ""
        if per_model:
            fused_choice = per_model[0]
            for nxt in per_model[1:]:
                fused_choice = fuse_texts(
                    fused_choice,
                    nxt,
                    prefer="a",
                    tie_badness_thresh=0.35,
                    w_a=0.55,
                    w_b=0.45,
                )

        shortlist = self.mbr._dedup([mbr_choice, fused_choice] + per_model + prepared[:8])
        if not shortlist:
            return self.cfg.default_translation

        try:
            chosen = self.mbr.pick(shortlist)
        except Exception as e:
            self.logger.warning(f"Hybrid shortlist MBR failed; using heuristic fallback: {e}")
            chosen = self._best_effort_pick(shortlist)

        return chosen or self._best_effort_pick(shortlist) or self.cfg.default_translation

    def _save_submission(self, rows: List[Tuple[str, str]], filename: str):
        out_path = Path(self.cfg.output_dir) / filename
        df = pd.DataFrame(rows, columns=["id", "translation"])
        df["translation"] = df["translation"].fillna("").map(
            lambda x: InferenceEngine._final_fix_one(str(x), min_words=self.cfg.min_words_fallback)
        )
        df.to_csv(out_path, index=False)
        self.logger.info(f"Saved submission → {out_path}")
        return df

    def run(self, test_df: pd.DataFrame, model_specs: List[ModelRunSpec]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
        cfg, logger = self.cfg, self.logger

        logger.info("=" * 60)
        logger.info("Ensemble × MBR | Cross-model candidate pooling")
        logger.info(f"Models: {', '.join(spec.label for spec in model_specs)}")
        logger.info(f"Pool/model: beam×{cfg.num_beam_cands} + sample×{cfg.num_sample_cands}")
        logger.info("=" * 60)

        dataset = AkkadianDataset(test_df, self.preprocessor)
        sample_ids = [str(s) for s in dataset.ids]

        all_pools: Dict[str, Dict[str, List[str]]] = {}
        model_best_by_label: Dict[str, Dict[str, str]] = {}
        saved_frames: Dict[str, pd.DataFrame] = {}

        for spec in model_specs:
            logger.info(f"Running model spec: {spec.label} | prompt={spec.prompt_template!r}")
            wrapper = ModelWrapper(spec.model_path, spec.cfg, logger, spec.label, prompt_template=spec.prompt_template)
            pools = self._run_one_model(wrapper, dataset)

            rows = []
            best_map = {}
            for sid in sample_ids:
                chosen = self._pick_for_submission(pools.get(sid, []) or [])
                rows.append((sid, chosen))
                best_map[sid] = chosen

            submission_name = spec.submission_name or f"submission_{slugify(spec.label)}.csv"
            saved_frames[spec.label] = self._save_submission(rows, submission_name)
            all_pools[spec.label] = pools
            model_best_by_label[spec.label] = best_map

            wrapper.unload()
            del wrapper
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

        logger.info(f"Phase {len(model_specs) + 1}/{len(model_specs) + 1} — hybrid MBR selection")
        results = []

        for sid in tqdm(sample_ids, desc="Hybrid-MBR"):
            combined = []
            per_model_best = []
            for spec in model_specs:
                combined.extend(all_pools.get(spec.label, {}).get(sid, []) or [])
                best_txt = model_best_by_label.get(spec.label, {}).get(sid, "")
                if best_txt:
                    per_model_best.append(best_txt)

            chosen = self._hybrid_pick(combined, per_model_best)
            results.append((sid, chosen))

            if cfg.checkpoint_freq and len(results) % cfg.checkpoint_freq == 0:
                ckpt = Path(cfg.output_dir) / f"checkpoint_{len(results)}.csv"
                pd.DataFrame(results, columns=["id", "translation"]).to_csv(ckpt, index=False)
                logger.info(f"Checkpoint saved → {ckpt}")

        df = pd.DataFrame(results, columns=["id", "translation"])
        self._validate(df)
        return df, saved_frames

    def _validate(self, df: pd.DataFrame):
        logger = self.logger
        logger.info("=" * 60)

        if df.empty:
            logger.warning("Validation skipped because the ensemble output is empty.")
            logger.info("=" * 60)
            return

        empty = df["translation"].fillna("").str.strip().eq("").sum()
        lens = df["translation"].fillna("").str.split().map(len)

        logger.info(
            f"Empty: {empty} ({100 * empty / max(1, len(df)):.2f}%) | "
            f"Len mean/med/min/max: {lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()}"
        )

        sample_idxs = sorted(set([0, len(df)//4, len(df)//2, 3*len(df)//4, len(df)-1]))
        for idx in sample_idxs:
            row = df.iloc[idx]
            logger.info(f"ID {row['id']}: {str(row['translation'])[:80]}")

        logger.info("=" * 60)


In [16]:
class InferenceEngine:
    def __init__(self, cfg: CFG):
        # FIX: keep the runtime config instance (not the CFG class itself)
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.preprocessor = Preprocessor()

        # Safe Postpreprocessor
        self.safe_pp = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=False)
        self.safe_pp_stripdash = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=True)
        # Aggressive Postprocessor
        self.agg_pp = AggressivePostprocessor()

        # Tokenizer + model
        path = cfg.model_path
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(path).to(self.device)
        self.model.eval()

        # Optional BetterTransformer
        if cfg.use_better_transformer and cfg.device == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info("BetterTransformer enabled.")
            except Exception as e:
                logger.warning(f"BetterTransformer failed, continue without it: {e}")

    def _collate_fn(self, batch):
        ids, texts = zip(*batch)
        enc = self.tokenizer(
            list(texts),
            padding=True,
            truncation=True,  # FIX: typo (trunctation -> truncation)
            max_length=self.cfg.max_length,
            return_tensors="pt"
        )
        return list(ids), enc

    def _adaptive_beams(self, attention_mask: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            # FIX: return beam count, not boolean flag
            return self.cfg.num_beams

        lens = attention_mask.sum(dim=1).detach().cpu().numpy()
        med = float(np.median(lens)) if len(lens) else 0.0
        if med < 100:
            return max(4, self.cfg.num_beams // 2)
        return self.cfg.num_beams

    def _postprocess(self, texts: List[str]) -> List[str]:
        mode = self.cfg.postprocess_mode
        if mode == "safe_only":
            return self.safe_pp.postprocess_batch(texts)
        if mode == "aggressive_only":
            out = self.safe_pp_stripdash.postprocess_batch(texts)
            return self.agg_pp.postprocess_batch(out)

        safe = self.safe_pp.postprocess_batch(texts)
        thr = self.cfg.aggressive_trigger_badness
        refined = []
        for t in safe:
            refined.append(self.agg_pp.postprocess_one(t) if badness_score(t) >= thr else t)
        return refined

    @staticmethod
    def _final_fix_one(t: str, min_words=3) -> str:
        tt = (t or "").strip()
        if not tt:
            return "The tablet contains fragmentary text."

        words = tt.split()
        if len(words) < min_words:
            return "The tablet contains an incomplete inscription."

        # FIX: call islower() and use tt[0].upper() (not tt[0].upper)
        if tt and tt[0].isalpha() and tt[0].islower():
            tt = tt[0].upper() + tt[1:]

        if not tt.endswith((".", "!", "?")):
            tt = tt + "."

        tt = re.sub(r"\s+([,.;:!?])", r"", tt)
        tt = re.sub(r"\s+", " ", tt).strip()
        return tt

    def generate_candidates(self, input_ids, attention_mask, beam_size):
        cfg = self.cfg
        bsz = input_ids.size(0)
    
        # ---- BEAM SEARCH ----
        beam_out = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=beam_size,
            num_return_sequences=cfg.num_beam_cands,
            max_new_tokens=cfg.max_new_tokens,
            length_penalty=cfg.length_penalty,
            early_stopping=cfg.early_stopping,
            no_repeat_ngram_size=cfg.no_repeat_ngram_size,
            do_sample=False,
        )
    
        # ---- SAMPLING ----
        sample_out = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=True,
            top_p=cfg.top_p,
            temperature=cfg.temperature,
            num_return_sequences=cfg.num_sample_cands,
            max_new_tokens=cfg.max_new_tokens,
        )
    
        # ---- DECODE ----
        beam_texts = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)
        sample_texts = self.tokenizer.batch_decode(sample_out, skip_special_tokens=True)
    
        # ---- GROUP BACK PER SAMPLE ----
        pools = []
        for i in range(bsz):
            pool = []
    
            # beam candidates
            for j in range(cfg.num_beam_cands):
                pool.append(beam_texts[i * cfg.num_beam_cands + j])
    
            # sample candidates
            for j in range(cfg.num_sample_cands):
                pool.append(sample_texts[i * cfg.num_sample_cands + j])
    
            pools.append(pool)
    
        return pools

    def run_inference(self, test_df: pd.DataFrame, run_tag="run") -> pd.DataFrame:
        cfg = self.cfg
        ds = AkkadianDataset(test_df, self.preprocessor)

        if cfg.use_bucket_batching:
            sampler = BucketSampler(ds.inputs, batch_size=cfg.batch_size, num_buckets=32)
            dl = DataLoader(
                ds,
                batch_sampler=sampler,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                collate_fn=self._collate_fn
            )
        else:
            dl = DataLoader(
                ds,  # FIX: df -> ds
                batch_size=cfg.batch_size,
                shuffle=False,
                num_workers=cfg.num_workers,
                pin_memory=cfg.pin_memory,
                collate_fn=self._collate_fn
            )

        results: List[Tuple[str, str]] = []
        t0 = time.time()

        if cfg.use_mixed_precision and cfg.device == "cuda":
            autocast_ctx = torch.cuda.amp.autocast
        else:
            class _NullCtx:
                def __enter__(self): return None
                def __exit__(self, exc_type, exc, tb): return False
            autocast_ctx = lambda: _NullCtx()

        for step, (ids, enc) in enumerate(dl):
            input_ids = enc["input_ids"].to(self.device, non_blocking=True)
            attn = enc["attention_mask"].to(self.device, non_blocking=True)

            beams = self._adaptive_beams(attn)
            gen_kwargs = dict(
                num_beams=beams,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                no_repeat_ngram_size=cfg.no_repeat_ngram_size
            )
            if cfg.repetition_penalty is not None:
                gen_kwargs["repetition_penalty"] = cfg.repetition_penalty

            with torch.inference_mode():
                with autocast_ctx():
                    out_ids = self.model.generate(
                        input_ids=input_ids,
                        attention_mask=attn,
                        **gen_kwargs
                    )

            # FIX: very important -> remove special tokens during decode
            decoded = self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)
            processed = self._postprocess(decoded)
            processed = [self._final_fix_one(x, cfg.min_words_fallback) for x in processed]
            results.extend(list(zip(ids, processed)))

            if cfg.checkpoint_freq and (len(results) % cfg.checkpoint_freq == 0):
                ck = pd.DataFrame(results, columns=["id", "translation"])
                ck_path = os.path.join(cfg.output_dir, f"checkpoint_{run_tag}_{len(results)}.csv")
                ck.to_csv(ck_path, index=False)
                logger.info(f"Saved checkpoint: {ck_path}")

            if cfg.device == "cuda" and cfg.empty_cache_every and (step + 1) % cfg.empty_cache_every == 0:
                torch.cuda.empty_cache()

            if (step + 1) % 50 == 0:
                elapsed = time.time() - t0
                logger.info(f"[{run_tag}] step={step+1}/{len(dl)} | done={len(results)} | elapsed={elapsed:.1f}s")

        df_out = pd.DataFrame(results, columns=["id", "translation"])
        self._validate(df_out, run_tag=run_tag)
        return df_out

    def _validate(self, df: pd.DataFrame, run_tag: str):
        if df.empty:
            logger.warning(f"[{run_tag}] Empty output dataframe.")
            return

        lens = df["translation"].fillna("").map(lambda x: len(str(x).split()))
        empties = (df["translation"].fillna("").str.strip() == "").mean() * 100
        short = (lens < 5).sum()

        logger.info(
            f"[{run_tag}] rows={len(df)} | empty%={empties:.2f} | "
            f"len(mean/med/min/max)={lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()} | "
            f"short(<5)={short}"
        )

        k = min(self.cfg.validate_samples, len(df))
        if k > 0:
            sample = df.sample(k, random_state=123)
            for _, r in sample.iterrows():
                logger.info(f"[{run_tag}] sample id={r['id']} | {str(r['translation'])[:160]}")



In [17]:
def make_cfg(base: CFG, preset: str, batch_size: Optional[int]=None, num_beams: Optional[int]=None, length_penalty: Optional[float]=None, repetition_penalty: Optional[float]=None, no_repeat_ngram_size: Optional[int]=None, strip_dash_old: Optional[bool]=None) -> CFG:
    cfg = CFG(**asdict(base))
    if preset == "baseline":
        cfg.length_penalty = 1.30
        cfg.repetition_penalty = None
    elif preset == "len115":
        cfg.length_penalty = 1.15
        cfg.repetition_penalty = None
    elif preset == "rep_pen":
        cfg.length_penalty = 1.30
        cfg.repetition_penalty = 1.08
    elif preset == "len115_rep":
        cfg.length_penalty = 1.15
        cfg.repetition_penalty = 1.08

    if batch_size is not None:
        cfg.batch_size = batch_size
    if num_beams is not None:
        cfg.num_beams = num_beams
    if length_penalty is not None:
        cfg.length_penalty = length_penalty
    if repetition_penalty is not None:
        cfg.repetition_penalty = repetition_penalty
    if no_repeat_ngram_size is not None:
        cfg.no_repeat_ngram_size = no_repeat_ngram_size
    return cfg


def fuse_submissions(df_a: pd.DataFrame, df_b: pd.DataFrame, prefer="a", tie_badness_thresh=0.5, w_a=0.60, w_b=0.40) -> pd.DataFrame:
    a = df_a.set_index("id")["translation"]
    b = df_b.set_index("id")["translation"]
    ids = a.index

    out = []

    for _id in ids:
        ta = a.loc[_id]
        tb = b.loc[_id]
        fused = fuse_texts(
            ta,
            tb,
            prefer=prefer,
            tie_badness_thresh=tie_badness_thresh,
            w_a=w_a,
            w_b=w_b
        )
        out.append((_id, fused))
    return pd.DataFrame(out, columns=["id", "translation"])


def setup_logging(output_dir):
    os.makedirs(output_dir, exist_ok=True)

    logger = logging.getLogger("inference")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    ch = logging.StreamHandler()
    ch.setFormatter(formatter)
    logger.addHandler(ch)

    log_path = os.path.join(output_dir, f"log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
    fh = logging.FileHandler(log_path)
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging to {log_path}")
    return logger


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(text).lower()).strip("-") or "model"


def resolve_model_path(path_hint: str, extra_globs: Optional[List[str]] = None) -> str:
    if path_hint and os.path.exists(path_hint):
        return path_hint

    patterns = [p for p in [path_hint, *(extra_globs or [])] if p]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern))
        if not pattern.startswith("/"):
            candidates.extend(glob.glob(os.path.join("/kaggle/input", pattern)))
            candidates.extend(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))

    if path_hint:
        base = os.path.basename(path_hint.rstrip("/"))
        candidates.extend(glob.glob(f"/kaggle/input/**/{base}", recursive=True))

    scored = []
    for cand in dict.fromkeys(candidates):
        if not os.path.isdir(cand):
            continue
        score = 0
        if os.path.exists(os.path.join(cand, "config.json")):
            score += 3
        if glob.glob(os.path.join(cand, "*.safetensors")) or glob.glob(os.path.join(cand, "pytorch_model*")):
            score += 3
        if os.path.exists(os.path.join(cand, "tokenizer_config.json")):
            score += 1
        if score:
            scored.append((score, cand))

    if scored:
        scored.sort(key=lambda x: (-x[0], len(x[1])))
        return scored[0][1]

    raise FileNotFoundError(f"Could not resolve model path from hint={path_hint!r} globs={extra_globs!r}")


def build_model_specs(base_cfg: CFG) -> List[ModelRunSpec]:
    specs = [
        ModelRunSpec(
            label="baseline_local",
            model_path=resolve_model_path(
                base_cfg.model_path,
                extra_globs=[
                    "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x",
                    "/kaggle/input/**/byt5-akkadian-optimized-34x",
                ],
            ),
            prompt_template="translate Akkadian to English: {text}",
            preset="len115",
            submission_name="submission_model_baseline_local.csv",
        ),
        ModelRunSpec(
            label="mattia_byt5_mbr_v2",
            model_path=resolve_model_path(
                "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
                extra_globs=[
                    "/kaggle/input/byt5-akkadian-mbr-v2/**",
                    "/kaggle/input/**/byt5-akkadian-mbr-v2/**",
                ],
            ),
            prompt_template="{text}",
            preset="rep_pen",
            submission_name="submission_model_mattia_byt5_mbr_v2.csv",
        ),
        ModelRunSpec(
            label="mattia_byt5_mbr_v6",
            model_path=resolve_model_path(
                "/kaggle/input/datasets/spencevanasperdt/mattiaangelibyt5-akkadian-mbrpytorchdefault6",
                extra_globs=[
                    "/kaggle/input/mattiaangelibyt5-akkadian-mbrpytorchdefault6/**",
                    "/kaggle/input/**/mattiaangelibyt5-akkadian-mbrpytorchdefault6/**",
                ],
            ),
            prompt_template="{text}",
            preset="len115_rep",
            submission_name="submission_model_mattia_byt5_mbr_v6.csv",
        ),
    ]

    materialized = []
    for spec in specs:
        spec_cfg = make_cfg(base_cfg, preset=spec.preset)
        spec_cfg.model_path = spec.model_path
        setattr(spec, "cfg", spec_cfg)
        materialized.append(spec)
    return materialized


## Helpers to build configs (two-pass)

## Main: load test, run 2 passes, fuse, save

In [18]:
base_cfg = CFG()
os.makedirs(base_cfg.output_dir, exist_ok=True)
os.makedirs(base_cfg.temp_model_dir, exist_ok=True)

logger = setup_logging(base_cfg.output_dir)

# =========================
# LOAD DATA
# =========================
test_df = pd.read_csv(base_cfg.test_path)

assert "id" in test_df.columns and "transliteration" in test_df.columns

logger.info(f"Loaded test rows={len(test_df)} from {base_cfg.test_path}")
if len(test_df) <= 10:
    logger.warning("Very small test set detected. Check test_path!")

model_specs = build_model_specs(base_cfg)
logger.info("Resolved ensemble members:")
for spec in model_specs:
    logger.info(
        f"  - {spec.label}: path={spec.model_path} | prompt={spec.prompt_template!r} | "
        f"preset={spec.preset} | submission={spec.submission_name}"
    )

with open(os.path.join(base_cfg.output_dir, "run_config.json"), "w") as f:
    json.dump(
        {
            **asdict(base_cfg),
            "ensemble_members": [
                {
                    "label": spec.label,
                    "model_path": spec.model_path,
                    "prompt_template": spec.prompt_template,
                    "preset": spec.preset,
                    "submission_name": spec.submission_name,
                }
                for spec in model_specs
            ],
        },
        f,
        indent=2,
    )

engine = EnsembleMBREngine(base_cfg, logger)
final_df, model_submissions = engine.run(test_df, model_specs)

final_df["translation"] = final_df["translation"].fillna("").map(
    lambda x: InferenceEngine._final_fix_one(
        str(x),
        min_words=base_cfg.min_words_fallback
    )
)

out_path = os.path.join(base_cfg.output_dir, "submission.csv")
final_df.to_csv(out_path, index=False)
logger.info(f"Saved FINAL submission: {out_path}")

for spec in model_specs:
    csv_name = spec.submission_name or f"submission_{slugify(spec.label)}.csv"
    logger.info(f"Per-model submission ready: {os.path.join(base_cfg.output_dir, csv_name)}")

lens = final_df["translation"].map(lambda x: len(str(x).split()))
logger.info(
    f"FINAL | rows={len(final_df)} | "
    f"len(mean/med/min/max)={lens.mean():.1f}/"
    f"{lens.median():.1f}/{lens.min()}/{lens.max()}"
)

print(final_df.head())
print(final_df.tail())


2026-03-22 13:15:47,598 | INFO | Logging to /kaggle/working/log_20260322_131547.txt
2026-03-22 13:15:47,598 | INFO | Logging to /kaggle/working/log_20260322_131547.txt
2026-03-22 13:15:47,621 | INFO | Loaded test rows=4 from /kaggle/input/deep-past-initiative-machine-translation/test.csv
2026-03-22 13:15:47,621 | INFO | Loaded test rows=4 from /kaggle/input/deep-past-initiative-machine-translation/test.csv
2026-03-22 13:15:47,623 | WARNING | Very small test set detected. Check test_path!
2026-03-22 13:15:47,623 | WARNING | Very small test set detected. Check test_path!
2026-03-22 13:15:47,627 | INFO | Resolved ensemble members:
2026-03-22 13:15:47,627 | INFO | Resolved ensemble members:
2026-03-22 13:15:47,628 | INFO |   - baseline_local: path=/kaggle/input/final-byt5/byt5-akkadian-optimized-34x | prompt='translate Akkadian to English: {text}' | preset=len115 | submission=submission_model_baseline_local.csv
2026-03-22 13:15:47,628 | INFO |   - baseline_local: path=/kaggle/input/final-b

[baseline_local]:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_55/2491571720.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ctx():
2026-03-22 13:16:45,492 | INFO | Saved submission → /kaggle/working/submission_model_baseline_local.csv
2026-03-22 13:16:45,492 | INFO | Saved submission → /kaggle/working/submission_model_baseline_local.csv
2026-03-22 13:16:56,118 | INFO | [baseline_local] Unloaded. GPU free: 15.63 GB
2026-03-22 13:16:56,118 | INFO | [baseline_local] Unloaded. GPU free: 15.63 GB
2026-03-22 13:16:56,430 | INFO | Running model spec: mattia_byt5_mbr_v2 | prompt='{text}'
2026-03-22 13:16:56,430 | INFO | Running model spec: mattia_byt5_mbr_v2 | prompt='{text}'
2026-03-22 13:16:56,431 | INFO | [mattia_byt5_mbr_v2] Loading from /kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1
2026-03-22 13:16:56,431 | INFO | [mattia_byt5_mbr_v2] Loading from /kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1
2

[mattia_byt5_mbr_v2]:   0%|          | 0/4 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/TensorCompare.cu:112: _assert_async_cuda_kernel: block: [0,0,0], thread: [0,0,0] Assertion `probability tensor contains either `inf`, `nan` or element < 0` failed.


AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
